# Chapter 20 — The Embedding Bridge

**Book alignment:** Embeddings From First Principles, Chapter 20

**Question this notebook isolates:** A map is a function; a **bridge** is a map plus the
record of *what it was shown to preserve, for whom*, and an explicit scope of what it must
not be used for. Given a Procrustes bridge's measured preservation profile (Wave 3,
MiniLM-L6 → mpnet-base), does `usable_for` come out to include retrieval while
`not_usable_for` includes threshold transfer and relation tasks?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. Build `usable_for` from the measured profile

In [ ]:
prof = art("wave3", "ladder-8property-matrix")["pairs"]["minilm-l6 vs mpnet-base"]["rungs"]["procrustes"]
keep = ("neighborhood_at10", "relation_profile_corr", "calibration_transfer",
        "hard_negative_ratio", "rank_triplet_agreement")
for k in keep:
    print(f"  {k:24} {prof[k]:.3f}")

BARS = {"retrieval":          ("neighborhood_at10",     0.65),
        "clustering":         ("relation_profile_corr", 0.80),
        "threshold_transfer": ("calibration_transfer",  0.90),
        "relation_tasks":     ("hard_negative_ratio",   0.70)}

usable, not_usable = [], []
for task, (metric, bar) in BARS.items():
    tag = f"{task} ({metric}={prof[metric]:.2f} vs {bar})"
    (usable if prof[metric] >= bar else not_usable).append(tag)

print("\nusable_for    :", usable)
print("not_usable_for:", not_usable)
assert any("retrieval" in u for u in usable)
assert any("threshold_transfer" in n for n in not_usable)
assert any("relation_tasks" in n for n in not_usable)

## 2. A consumer asks the bridge — and gets a scoped answer, not "spaces are compatible"

In [ ]:
def bridge_permits(operation):
    metric, bar = BARS[operation]
    val = prof[metric]
    return ("YES" if val >= bar else "NO", f"{metric}={val:.2f} (bar {bar})")

for op in ("retrieval", "threshold_transfer", "relation_tasks"):
    verdict, why = bridge_permits(op)
    print(f"  can I use you for {op:20}? {verdict:3}  [{why}]")
assert bridge_permits("retrieval")[0] == "YES"
assert bridge_permits("relation_tasks")[0] == "NO"
print("\ncompatibility is a per-task vector - a bridge can preserve retrieval while destroying calibration")

## What we earned

A bridge carries directional source/target `space_hash` values, a method, a *reconstruction*
metric family and a *preservation* metric family, cross-space calibration, and an explicit
`usable_for` / `not_usable_for`. Every `YES` has a measured metric above a stated bar; every
`NO` has a one-line reason. Bridges are directional, compose lossily, and are pinned to
exact hashes.

**Notebook 21 / Chapter 21** is the empirical investigation behind those preservation
numbers — and shows a bridge can *invert* the polarity distinction.